# 🎬 박스오피스 데이터 수집
## 1. 주요 기능
- **기간 단위 수집**: 시작일(`start_dt`)과 종료일(`end_dt`)을 지정하여 특정 기간의 데이터를 일괄 수집합니다.
- **Bulk Insert (대량 저장)**: DB 부하를 줄이기 위해 매일 저장하지 않고 데이터를 모아서 한꺼번에 저장합니다.
- **중간 저장 (Checkpoint)**: 대량 수집 중 오류가 발생할 경우를 대비하여 **30일 단위로 자동 저장**을 수행합니다.
- **시장 통계 자동 집계**: 박스오피스 상위 10개 영화 데이터를 바탕으로 당일 전체 시장 규모(관객수, 매출액, 1위 점유율 등)를 계산하여 별도 테이블(`daily_market_stats`)에 저장합니다.

## 2. 사용 방법

노트북의 마지막 셀에서 다음과 같이 호출하여 사용합니다.

```python
from datetime import date

# 수집할 기간 설정 (예: 2023년 전체)
start = date(2023, 1, 1)
end = date(2023, 12, 31)

# 수집 실행
fetch_and_save_box_office_range(start, end)
```

## 3. 주의 사항
- **API 한도**: KOBIS API는 키당 하루 **3,000회** 호출 제한이 있습니다.


In [2]:
from datetime import date, timedelta
from data.api import get_daily_box_office
from data.db import db

준비된 데이터를 실제 DB 테이블에 삽입합니다.
- **Upsert 전략**: `ON DUPLICATE KEY UPDATE` 구문을 사용하여 이미 존재하는 날짜/영화 데이터는 최신본으로 업데이트(Overwrite)합니다.
- **예외 처리**: DB 실행 중 오류 발생 시 `RuntimeError`를 던져 작업을 중단시키고 에러 원인을 명확히 표시합니다.

In [3]:
def bulk_insert_box_office(rows):
    """박스오피스 상세 데이터를 Bulk Insert 합니다."""
    query = """
    INSERT INTO daily_box_office (
        target_date, movie_id, boxofficeType, showRange, `rank`, rankInten, rankOldAndNew,
        salesAmt, salesShare, salesInten, salesChange, salesAcc,
        audiCnt, audiInten, audiChange, audiAcc, scrnCnt, showCnt
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        `rank` = VALUES(`rank`),
        rankInten = VALUES(rankInten),
        rankOldAndNew = VALUES(rankOldAndNew),
        salesAmt = VALUES(salesAmt),
        salesShare = VALUES(salesShare),
        salesInten = VALUES(salesInten),
        salesChange = VALUES(salesChange),
        salesAcc = VALUES(salesAcc),
        audiCnt = VALUES(audiCnt),
        audiInten = VALUES(audiInten),
        audiChange = VALUES(audiChange),
        audiAcc = VALUES(audiAcc),
        scrnCnt = VALUES(scrnCnt),
        showCnt = VALUES(showCnt)
    """
    affected = db.execute_many(query, rows)
    print(f"✅ DB 저장 완료: daily_box_office ({len(rows)}건 처리)")
    return affected

def bulk_insert_market_stats(rows):
    """시장 통계 데이터를 Bulk Insert 합니다."""
    query = """
    INSERT INTO daily_market_stats (
        target_date, total_audi, total_show, top1_share, total_sales, total_scrn
    ) VALUES (%s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        total_audi = VALUES(total_audi),
        total_show = VALUES(total_show),
        top1_share = VALUES(top1_share),
        total_sales = VALUES(total_sales),
        total_scrn = VALUES(total_scrn)
    """
    affected = db.execute_many(query, rows)

    print(f"✅ DB 저장 완료: daily_market_stats ({len(rows)}건 처리)")
    return affected


당일 박스오피스 결과를 분석하여 시장 통계를 생성합니다.
- **집계 항목**: 전체 관객수, 전체 상영횟수, 전체 매출액, 전체 스크린 수, 1위 영화 점유율.

In [4]:
def calculate_market_stats_row(target_dt_db: str, result):
    """
    박스오피스 결과를 바탕으로 당일 시장 통계 튜플을 생성합니다.
    """
    total_audi = 0
    total_show = 0
    total_sales = 0
    total_scrn = 0
    top1_share = 0.0

    for item in result.dailyBoxOfficeList:
        total_audi += item.audiCnt
        total_show += item.showCnt
        total_sales += item.salesAmt
        total_scrn += item.scrnCnt
        
        if item.rank == 1:
            top1_share = item.salesShare
            
    return (target_dt_db, total_audi, total_show, top1_share, total_sales, total_scrn)

수집 프로세스를 관리하는 메인 함수입니다.
- **작동 방식**:
    1. `start_dt`부터 `end_dt`까지 1일씩 증가하며 KOBIS API를 호출합니다.
    2. 응답받은 데이터를 전처리하여 `all_box_office_rows` 리스트에 담습니다.
    3. `success_days`가 **30의 배수**가 될 때마다 `_bulk_insert` 함수를 호출하여 DB에 중간 저장합니다.
    4. 루프 종료 후 남은 데이터를 최종 저장합니다.

In [5]:
def fetch_and_save_box_office_range(start_dt: date, end_dt: date):
    """
    지정된 시작일부터 종료일까지의 박스오피스 데이터를 순차적으로 수집한 후,
    DB에 한 번에 Bulk로 저장합니다.
    
    Args:
        start_dt (date): 시작 날짜
        end_dt (date): 종료 날짜
        
    Returns:
        int: 총 성공적으로 처리된 일수
    """
    current_dt = start_dt
    all_box_office_rows = []
    all_market_stats_rows = []
    success_days = 0
    
    print(f"[{start_dt} ~ {end_dt}] 기간 데이터 수집 시작 (Bulk 처리 모드)...")
    
    while current_dt <= end_dt:
        # 1. API 호출
        response = get_daily_box_office(current_dt)
        if not response or not response.boxOfficeResult:
            print(f"[{current_dt}] API 응답이 없거나 결과가 비어 있습니다. (건너뜀)")
        else:
            result = response.boxOfficeResult
            target_date_db = current_dt.isoformat()
            
            # 2. 박스오피스 상세 리스트 데이터 수집
            for item in result.dailyBoxOfficeList:
                all_box_office_rows.append((
                    target_date_db, item.movieCd, result.boxofficeType, result.showRange,
                    item.rank, item.rankInten, item.rankOldAndNew,
                    item.salesAmt, item.salesShare, item.salesInten, item.salesChange, item.salesAcc,
                    item.audiCnt, item.audiInten, item.audiChange, item.audiAcc, item.scrnCnt, item.showCnt
                ))
            
            # 3. 시장 통계 데이터 집계 (리스트에 보관)
            market_row = calculate_market_stats_row(target_date_db, result)
            all_market_stats_rows.append(market_row)
            success_days += 1
            
            # 진행 상태 로그
            if success_days % 30 == 0:
                print(f" > {current_dt} 까지 수집 및 중간 저장 중... ({success_days}일치)")
                # 30일마다 중간 저장 (안전 장치)
                bulk_insert_box_office(all_box_office_rows)
                bulk_insert_market_stats(all_market_stats_rows)
                all_box_office_rows.clear()
                all_market_stats_rows.clear()
            elif success_days % 10 == 0:
                print(f" > {current_dt} 수집 중...")

        current_dt += timedelta(days=1)

    # 4. 마지막 남은 데이터 저장 (Bulk Insert)
    if all_box_office_rows:
        bulk_insert_box_office(all_box_office_rows)
    
    if all_market_stats_rows:
        bulk_insert_market_stats(all_market_stats_rows)

    print(f"\n✅ 전체 수집 및 저장 완료: 총 {success_days}일 분량의 데이터가 처리되었습니다.")
    return success_days

In [ ]:
# fetch_and_save_box_office_range(date(2023,1,1), date(2023,12,31))

[2023-01-01 ~ 2023-12-31] 기간 데이터 수집 시작 (Bulk 처리 모드)...
 > 2023-01-10 수집 중...
 > 2023-01-20 수집 중...
 > 2023-01-30 까지 수집 및 중간 저장 중... (30일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2023-02-09 수집 중...
 > 2023-02-19 수집 중...
 > 2023-03-01 까지 수집 및 중간 저장 중... (60일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2023-03-11 수집 중...
 > 2023-03-21 수집 중...
 > 2023-03-31 까지 수집 및 중간 저장 중... (90일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2023-04-10 수집 중...
 > 2023-04-20 수집 중...
 > 2023-04-30 까지 수집 및 중간 저장 중... (120일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2023-05-10 수집 중...
 > 2023-05-20 수집 중...
 > 2023-05-30 까지 수집 및 중간 저장 중... (150일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2023-06-09 수집 중...
 > 2023-06-19 수집 중...
 > 2023-06-29 까지 수집 및 중간 저장 중... (180일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB

365

In [ ]:
# fetch_and_save_box_office_range(date(2024,1,1), date(2024,12,31))

[2024-01-01 ~ 2024-12-31] 기간 데이터 수집 시작 (Bulk 처리 모드)...
 > 2024-01-10 수집 중...
 > 2024-01-20 수집 중...
 > 2024-01-30 까지 수집 및 중간 저장 중... (30일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2024-02-09 수집 중...
 > 2024-02-19 수집 중...
 > 2024-02-29 까지 수집 및 중간 저장 중... (60일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2024-03-10 수집 중...
 > 2024-03-20 수집 중...
 > 2024-03-30 까지 수집 및 중간 저장 중... (90일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2024-04-09 수집 중...
 > 2024-04-19 수집 중...
 > 2024-04-29 까지 수집 및 중간 저장 중... (120일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2024-05-09 수집 중...
 > 2024-05-19 수집 중...
 > 2024-05-29 까지 수집 및 중간 저장 중... (150일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2024-06-08 수집 중...
 > 2024-06-18 수집 중...
 > 2024-06-28 까지 수집 및 중간 저장 중... (180일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB

366

In [ ]:
# fetch_and_save_box_office_range(date(2025,1,1), date(2025,12,31))

[2025-01-01 ~ 2025-12-31] 기간 데이터 수집 시작 (Bulk 처리 모드)...
 > 2025-01-10 수집 중...
 > 2025-01-20 수집 중...
 > 2025-01-30 까지 수집 및 중간 저장 중... (30일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2025-02-09 수집 중...
 > 2025-02-19 수집 중...
 > 2025-03-01 까지 수집 및 중간 저장 중... (60일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2025-03-11 수집 중...
 > 2025-03-21 수집 중...
 > 2025-03-31 까지 수집 및 중간 저장 중... (90일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2025-04-10 수집 중...
 > 2025-04-20 수집 중...
 > 2025-04-30 까지 수집 및 중간 저장 중... (120일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2025-05-10 수집 중...
 > 2025-05-20 수집 중...
 > 2025-05-30 까지 수집 및 중간 저장 중... (150일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB 저장 완료: daily_market_stats (30건 처리)
 > 2025-06-09 수집 중...
 > 2025-06-19 수집 중...
 > 2025-06-29 까지 수집 및 중간 저장 중... (180일치)
✅ DB 저장 완료: daily_box_office (300건 처리)
✅ DB

365